# `evap_cool` — equilibrium thermodynamics

This notebook visualises the post-processed thermodynamics produced by
`evap_cool.post_processing`: the equilibrium state functions
$\Omega$, $S$, $P$, $H$, $F$, $G$ and the thermal coefficients
$C_V$, $C_P$, $\kappa_T$, $B_P$, evaluated at every step of the cooling
trajectory.

For each trap (box, quadrupole, oscillator) we plot two figures:

1. **State functions** — a 2×3 grid of $\Omega$, $S$, $P$, $H$, $F$, $G$
   vs sample temperature $T$.
2. **Thermal coefficients** — a 2×2 grid of $C_V$, $C_P$, $\kappa_T$,
   $B_P$ vs $T$.

Three statistics are overlaid on every panel: **Maxwell-Boltzmann**
(blue) as the classical baseline, **Bose-Einstein** (green), and
**Fermi-Dirac** (red).  In the non-degenerate regime ($\alpha \to -\infty$)
all three coincide; the gap that opens up between BE/FD and MB as $T$
drops is the quantum-degeneracy signal.  In that limit every polylog
ratio $g_{\nu+k}/g_\nu \to 1$ and the closed-form MB expressions from
`mb_limit_based.pdf` §5 apply:

$$
\Omega_\mathrm{MB} = -Nk_BT,\quad
S_\mathrm{MB} = Nk_B\!\bigl[(s{+}1)-\alpha\bigr],\quad
C_V^\mathrm{MB} = sNk_B,\quad
C_P^\mathrm{MB} = (s{+}1)Nk_B,\quad
\kappa_T^\mathrm{MB} = \tfrac{V_g}{Nk_BT},\quad
B_P^\mathrm{MB} = \tfrac{1}{T},
$$

with $s=3/2$ (box), $s=3$ (oscillator), $s=9/2$ (quadrupole).

> **Prerequisite.** Run `evap_cool_usage.ipynb` first to produce the
> source `*.json` files (`<trap>_bosons.json`, `<trap>_fermions.json`,
> `<trap>_mb.json`) and the BE/FD `*_thermo.json` files.  This notebook
> will *create* the MB `*_thermo.json` files on the fly when it doesn't
> find them, since the usage notebook's post-processing step currently
> only covers BE/FD.


## Setup

We import the post-processing helpers (`compute_mb_run_thermodynamics`
falls back when an MB thermo file is missing) and the plotting helpers
from `evap_cool.plots`:

- `plot_state_functions(T_b, thermo_b, T_f, thermo_f, trap_name,
                        T_mb=..., thermo_mb=..., ...)`
- `plot_thermal_coefficients(T_b, thermo_b, T_f, thermo_f, trap_name,
                             T_mb=..., thermo_mb=..., ...)`

Both default to log-log axes. Quantities that can be negative
($\Omega$, $F$, $G$) are plotted in absolute value with the title
showing $|\cdot|$ — set `log_y=False` if you'd rather see the signed
values on a linear scale.  The new `T_mb` / `thermo_mb` arguments
overlay the Maxwell-Boltzmann reference; omit them to recover the
B-vs-F-only behaviour of earlier versions.


In [ ]:
import matplotlib.pyplot as plt

from evap_cool import (
    load_run, load_thermodynamics,
    list_sessions, list_runs,
    process_and_save_mb_run, load_thermodynamics,
    plot_state_functions, plot_thermal_coefficients,
    BoxTrap, QuadrupoleTrap, OscillatorTrap,
)
import numpy as np


## Locate the most recent session

`list_sessions("runs")` returns every `runs/<date>/<time>/` folder
chronologically; we pick the latest. Override `session` manually if you
want to plot an older run.


In [ ]:
sessions = list_sessions("runs")
if not sessions:
    raise RuntimeError(
        "No sessions found under runs/. "
        "Run evap_cool_usage.ipynb first to generate the thermo files."
    )

session = sessions[-1]
print(f"Using session : {session}")
print(f"Files present :")
for p in list_runs(session):
    print(f"  {p.relative_to(session)}")


## Helper: load one trap's source + thermo files (B / F / MB)

For a given trap label (e.g. `"box"`, `"quadrupole"`, `"oscillator"`)
we need six files in the session folder:

- `<trap>_bosons.json`   + `<trap>_bosons_thermo.json`
- `<trap>_fermions.json` + `<trap>_fermions_thermo.json`
- `<trap>_mb.json`       + `<trap>_mb_thermo.json`

The MB *source* run (`<trap>_mb.json`) is produced by
`evap_cool_usage.ipynb`.  The MB *thermo* file is usually missing
because the usage notebook only post-processes BE/FD; the helper below
generates it on demand by calling `process_and_save_mb_run`, which
needs a reconstructed `Trap` instance — that's the second argument to
`load_thermo_pair`.

The helper returns `(T_b, thermo_b, T_f, thermo_f, T_mb, thermo_mb)`
ready to feed into the plot functions, or `None` if any required file
is missing (so we can gracefully skip traps without post-processed
data).


In [ ]:
def load_thermo_pair(session, trap_label, trap=None):
    """Return (T_b, thermo_b, T_f, thermo_f, T_mb, thermo_mb) or None.

    Parameters
    ----------
    session : Path
        Session folder containing the source and thermo JSONs.
    trap_label : str
        File-name stem, e.g. ``"box"``.
    trap : Trap, optional
        Trap instance compatible with the saved MB run.  Required only
        if the MB thermo file doesn't already exist on disk; it is
        passed straight to `process_and_save_mb_run` to generate it.
    """
    needed = {
        "src_b":  session / f"{trap_label}_bosons.json",
        "thr_b":  session / f"{trap_label}_bosons_thermo.json",
        "src_f":  session / f"{trap_label}_fermions.json",
        "thr_f":  session / f"{trap_label}_fermions_thermo.json",
        "src_mb": session / f"{trap_label}_mb.json",
    }
    missing = [p.name for p in needed.values() if not p.exists()]
    if missing:
        print(f"  [skip] {trap_label}: missing {missing}")
        return None

    # MB thermo file: post-process on the fly if it doesn't exist yet.
    thr_mb_path = session / f"{trap_label}_mb_thermo.json"
    if not thr_mb_path.exists():
        if trap is None:
            print(f"  [skip] {trap_label}: MB thermo missing and no `trap` given to build it")
            return None
        print(f"  [process] {trap_label}: generating {thr_mb_path.name}")
        process_and_save_mb_run(needed["src_mb"], trap)

    src_b  = load_run(needed["src_b"])
    src_f  = load_run(needed["src_f"])
    src_mb = load_run(needed["src_mb"])
    thr_b  = load_thermodynamics(needed["thr_b"])
    thr_f  = load_thermodynamics(needed["thr_f"])
    thr_mb = load_thermodynamics(thr_mb_path)

    return (
        src_b["results"]["T"],  thr_b["results"],
        src_f["results"]["T"],  thr_f["results"],
        src_mb["results"]["T"], thr_mb["results"],
    )


def units_for(trap_label):
    """Energy-unit string per trap. Box runs in SI (J); quadrupole and
    oscillator default to eV."""
    return "J" if trap_label == "box" else "eV"


## 1. Box trap — $s = 3/2$

The box uses SI units, so all energy-bearing axes are in J.  The MB
trajectory provides the classical reference: at the high-$T$ end of the
run BE / FD / MB coincide, and the BE / FD lines peel away as the gas
becomes degenerate.


In [ ]:
trap_box = BoxTrap(V=1e-11)   # must match the trap used in the upstream run
data = load_thermo_pair(session, "box", trap=trap_box)
if data is not None:
    T_b, thermo_b, T_f, thermo_f, T_mb, thermo_mb = data
    fig_box_state = plot_state_functions(
        T_b, thermo_b, T_f, thermo_f,
        trap_name="Box",
        T_mb=T_mb, thermo_mb=thermo_mb,
        units_energy=units_for("box"),
    )
    plt.show()


In [ ]:
if data is not None:
    fig_box_coef = plot_thermal_coefficients(
        T_b, thermo_b, T_f, thermo_f,
        trap_name="Box",
        T_mb=T_mb, thermo_mb=thermo_mb,
        units_energy=units_for("box"),
    )
    plt.show()


## 2. Quadrupole trap — $s = 9/2$

Linear potential $U(r) = \bar A\, r$, generalized volume $V_g = \bar A^{-3}$,
in eV units.  In the MB limit $C_V = (9/2)Nk_B$ and $C_P = (11/2)Nk_B$,
both linear in $N$ — visible as parallel straight lines (on log-log) at
high $T$.


In [ ]:
trap_quad = QuadrupoleTrap(A_bar=1e-15)
data = load_thermo_pair(session, "quadrupole", trap=trap_quad)
if data is not None:
    T_b, thermo_b, T_f, thermo_f, T_mb, thermo_mb = data
    fig_quad_state = plot_state_functions(
        T_b, thermo_b, T_f, thermo_f,
        trap_name="Quadrupole",
        T_mb=T_mb, thermo_mb=thermo_mb,
        units_energy=units_for("quadrupole"),
    )
    plt.show()


In [ ]:
if data is not None:
    fig_quad_coef = plot_thermal_coefficients(
        T_b, thermo_b, T_f, thermo_f,
        trap_name="Quadrupole",
        T_mb=T_mb, thermo_mb=thermo_mb,
        units_energy=units_for("quadrupole"),
    )
    plt.show()


## 3. Harmonic oscillator trap — $s = 3$

Isotropic 3D oscillator $U(r) = \tfrac{1}{2} m \omega^2 r^2$, $V_g = 1/\bar\omega^3$,
eV units.  The MB lines obey $E = 3Nk_BT$, $H = 4Nk_BT$, $C_V = 3Nk_B$,
$C_P = 4Nk_B$; BE / FD recover these at high $T$ and depart from them
as $\alpha \to 0^-$ (bosons, BEC vicinity) or $\alpha \to 0^+$ (fermions,
Fermi-degeneracy onset).


In [ ]:
trap_osc = OscillatorTrap(omega=2 * np.pi * 100)
data = load_thermo_pair(session, "oscillator", trap=trap_osc)
if data is not None:
    T_b, thermo_b, T_f, thermo_f, T_mb, thermo_mb = data
    fig_osc_state = plot_state_functions(
        T_b, thermo_b, T_f, thermo_f,
        trap_name="Oscillator",
        T_mb=T_mb, thermo_mb=thermo_mb,
        units_energy=units_for("oscillator"),
    )
    plt.show()


In [ ]:
if data is not None:
    fig_osc_coef = plot_thermal_coefficients(
        T_b, thermo_b, T_f, thermo_f,
        trap_name="Oscillator",
        T_mb=T_mb, thermo_mb=thermo_mb,
        units_energy=units_for("oscillator"),
    )
    plt.show()


## 4. Reading the figures

A few things worth noting on the plots:

- **Maxwell-Boltzmann baseline.** The blue MB curve is the
  *statistics-independent* classical limit ($\alpha \to -\infty$).  In
  this limit every polylog ratio $g_{\nu+k}/g_\nu$ collapses to 1, so
  state functions and coefficients take the closed forms in §5 of
  `mb_limit_based.pdf`.  At high $T$ (early in the evaporation), BE /
  FD / MB are visually indistinguishable; the **separation between
  BE / FD and MB at low $T$ is the quantum-degeneracy signature**.

- **Sign convention.** $\Omega = -PV$ is always negative.  $F = G + \Omega$
  and $G = \mu N$ change sign with the regime ($\mu < 0$ for bosons in
  the non-degenerate limit; fermions can have $\mu > 0$).  With the
  default `log_y=True`, all three are shown as $|\cdot|$ — and the MB
  limit gives $F_\mathrm{MB} = Nk_BT(\alpha-1)$, $G_\mathrm{MB} = Nk_BT\alpha$,
  both negative for $\alpha < 1$.  Pass `log_y=False` to either plot
  function to see signed values on a linear axis.

- **Bosons vs fermions split.** Within a single panel, the green/red
  ordering reflects how $g_s(\alpha)$ differs between BE and FD at the
  same $(N, T)$.  The gap typically widens as $T$ drops, mirroring the
  onset of degeneracy.  Bosons tend to deviate *below* the MB curve
  for thermodynamic quantities that grow with $\alpha$, fermions
  *above* — the sign flip captured by the $\pm$ in the polylog
  recurrence.

- **MB sanity checks (§6 of the limit notes).** Each MB panel should
  obey: $E/(sNk_BT) = 1$ (equipartition), $PV/(Nk_BT) = 1$ (ideal gas),
  $C_P - C_V = Nk_B$ (Mayer), $C_V$ and $C_P$ both flat-in-$T$ at fixed
  $N$.  Visible curvature of the *MB* lines on log-log axes therefore
  reflects $N$ changing along the evaporation, not any temperature
  dependence of the heat capacities.

- **Heat-capacity ratio.** $C_P > C_V$ is required by stability; the
  two should track each other.  A panel where $C_P$ falls below $C_V$
  is a numerical artefact (typically near a degeneracy boundary where
  $\kappa_T$ or $B_P$ blow up).

## 5. Tweaks

- `log_x=False` / `log_y=False` for linear axes (signed values, no abs).
- `n_b`, `n_f`, `n_mb` to trim each statistics' trajectory if you want
  to exclude the early (high-$T$, weakly-degenerate) or late
  (near-halt) parts.
- `T_mb=None, thermo_mb=None` removes the MB overlay (or simply omit
  both keywords — they default to `None`).
- `figsize=(W, H)` to resize.
- `units_energy="..."` if you've built a trap with a custom unit system.
